In [3]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import numpy as np
from sklearn.utils.class_weight import compute_class_weight


# Global namefiles used for read csv files

In [4]:
global_path_train = 'data/final_train_data_transmission.csv'
global_path_test  = 'data/final_test_data_transmission.csv'
global_path_val   = 'data/final_val_data_transmission.csv'

# Testing Models

## Logistic Regression, Random Forest, MLP

In [5]:
##filename = 'data/final_data_tranmissions_v5.csv'

df_train = pd.read_csv(global_path_train)
df_test  = pd.read_csv(global_path_test)
df_val   = pd.read_csv(global_path_val)

features = ['latDev', 'longDev', 'elevSat', 'loraTP', 'loraSF', 'doppler', 'alt', 'raan']

X_train_features = df_train[features].values
y_train          = df_train['rcvOk'].values

X_test_features = df_test[features].values
y_test          = df_test['rcvOk'].values

X_val_features = df_val[features].values
y_val          = df_val['rcvOk'].values

scaler = StandardScaler()
scaler.fit(np.concatenate((X_train_features, X_test_features, X_val_features), axis=0))

X_train = scaler.transform(X_train_features)
X_test  = scaler.transform(X_test_features)
X_val   = scaler.transform(X_val_features)

#X_scaled = scaler.fit_transform(np.concatenate((X_train, X_test), axis=0))
#X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

''' classes = np.unique(y_train)
weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
) 

class_weights = dict(zip(classes, weights))
print('class_weights: ', class_weights)
'''
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "MLP": MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=1000, random_state=42)
}

# Training and testing
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    auc = roc_auc_score(y_test, y_pred)

    print(f"Model: {name}")
    print(f"Accuracy:  {acc*100:.2f}%")
    print(f"Precision: {prec*100:.2f}%")
    print(f"Recall:    {rec*100:.2f}%")
    print(f"F1-Score:  {f1*100:.2f}%")
    print(f"AUC-ROC:  {auc:.4f}%\n")

Model: Logistic Regression
Accuracy:  80.33%
Precision: 83.41%
Recall:    71.40%
F1-Score:  76.94%
AUC-ROC:  0.7966%

Model: Random Forest
Accuracy:  88.25%
Precision: 85.15%
Recall:    90.15%
F1-Score:  87.58%
AUC-ROC:  0.8839%

Model: MLP
Accuracy:  88.60%
Precision: 89.78%
Recall:    84.85%
F1-Score:  87.24%
AUC-ROC:  0.8832%



In [ ]:
Model: Logistic Regression
Accuracy:  81.94%
Precision: 79.56%
Recall:    74.04%
F1-Score:  76.70%
AUC-ROC:  80.64%

Model: Random Forest
Accuracy:  91.59%
Precision: 87.73%
Recall:    91.90%
F1-Score:  89.77%
AUC-ROC:  91.64%

Model: MLP
Accuracy:  91.18%
Precision: 88.66%
Recall:    89.46%
F1-Score:  89.06%
AUC-ROC:  90.89%

Model: Logistic Regression
Accuracy:  94.62%
Precision: 77.35%
Recall:    98.70%
F1-Score:  86.73%

Model: Random Forest
Accuracy:  95.31%
Precision: 87.96%
Recall:    85.39%
F1-Score:  86.66%

Model: MLP
Accuracy:  96.24%
Precision: 88.82%
Recall:    90.26%
F1-Score:  89.53%   

## Bidirectional LSTM

In [6]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, random_split
import math
import torch.optim as optim
import itertools
import time
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [7]:
def load_and_prepare_data(seq_length=16):    
    print("Loading dataset...")

    df_train = pd.read_csv(global_path_train)
    df_test  = pd.read_csv(global_path_test)
    df_val   = pd.read_csv(global_path_val)

    # Selecting continuous features
    continuous_features = ['latDev', 'longDev', 'elevSat', 'loraTP', 'loraSF', 'doppler', 'alt', 'raan']
    
    X_train_features = df_train[continuous_features].values
    #y_train          = df_train['rcvOk'].values

    X_test_features = df_test[continuous_features].values
    #y_test          = df_test['rcvOk'].values

    X_val_features = df_val[continuous_features].values
    


    scaler = StandardScaler()
    scaler.fit(np.concatenate((X_train_features, X_test_features, X_val_features), axis=0))

    #X_train = scaler.transform(X_train_features)
    #X_test  = scaler.transform(X_test_features)

    def create_sequences_data(df, scaler):
        cols_to_drop = ['dstId', 'srcSat', 'dstSat', 'loraCF', 'loraBW', 'loraCR', 'satId', 'srcId']
        df           = df.drop(columns=cols_to_drop)

        # Sort by id_simulaiton and time
        df = df.sort_values(by=['id_simulation', 'time']).reset_index(drop=True)
        
        # Apply StandardScaler         
        df[continuous_features] = scaler.transform(df[continuous_features])

        # Process data
        seq_X   = []
        label_y = []
        for sim_id, group_df in df.groupby('id_simulation'):
            num_features_array  = group_df[continuous_features].values
            ###cat_features_array  = group_df['srcId'].values
            time_array          = group_df['time'].values
            target_array        = group_df['rcvOk'].values
            num_packets         = len(group_df)
            
            # Discard if the simulation has less than SEQ_LENGHTS
            if num_packets < seq_length:
                continue

            # Create sliding windows - Ventanas deslizantes
            for i in range(num_packets - seq_length + 1):
                # Select the window        
                window_num      = num_features_array[i : i + seq_length]
                ###window_cat      = cat_features_array[i : i + SEQ_LENGTH].reshape(-1, 1)
                window_times    = time_array[i : i + seq_length]
                window_targets  = target_array[i : i + seq_length]
                
                # El paquete central a predecir es el último de la ventana
                # The prediction should be the last elemento of the window
                target_idx      = seq_length - 1
                label_target    = window_targets[target_idx]
                time_target     = window_times[target_idx]
                            
                # Si un paquete ocurrió en el mismo segundo que el objetivo, delta_t = 0        
                delta_t = (window_times - time_target).reshape(-1, 1)
                #delta_t =  time_target - window_times
                #print('delta_t: ', delta_t)
                
                #window_X = np.hstack((window_num, window_cat, delta_t))
                window_X = np.hstack((window_num, delta_t))

                seq_X.append(window_X)
                label_y.append(label_target)
                    
        X = torch.tensor(np.array(seq_X), dtype=torch.float32)
        y = torch.tensor(np.array(label_y), dtype=torch.float32)

        return X, y
    

    X_train, y_train = create_sequences_data(df_train, scaler)
    X_test , y_test  = create_sequences_data(df_test, scaler)
    X_val  , y_val   = create_sequences_data(df_val, scaler)


    return X_train, y_train, X_test, y_test, X_val  , y_val

In [8]:
class LoraCollisionLSTM(nn.Module):
    def __init__(self, num_features=9, hidden_size=64, num_layers=2, dropout=0.1, is_bidirectional=True):
        super().__init__()
        
        # Bidirectional LSTM Bidireccional        
        self.lstm = nn.LSTM(
            input_size    = num_features, 
            hidden_size   = hidden_size, 
            num_layers    = num_layers, 
            batch_first   = True,
            dropout       = dropout,
            bidirectional = is_bidirectional
        )
        
        # factor dinámico
        direction_factor = 2 if is_bidirectional else 1

        # Classifier, similar to Transformer model        
        # Como es Bidireccional, genera el doble de características (hidden_size * 2) # porque concatena lo que aprendió yendo hacia adelante y hacia atrás.        
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * direction_factor, 32),            
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        # Doc oficial.
        # output: tensor of shape (L,D∗Hout)(L,D∗Hout​) for unbatched input, (L,N,D∗Hout)(L,N,D∗Hout​) when batch_first=False 
        # or (N,L,D∗Hout)(N,L,D∗Hout​) when batch_first=True containing the output features (h_t) from the last layer of the LSTM, for each t.
        lstm_out, (hn, cn) = self.lstm(x)
        # lstm_out shape: (Batch, Seq_Length, hidden_size * 2)
        
        # Extraer la conclusión del paquete objetivo (el último de la ventana temporal)
        # El primer : mantiene todos los elementos del batch N
        # El -1 se enfoca en el ultimo elemenot de la secuencia
        # es el hidden size x 2
        target_packet = lstm_out[:, -1, :] # Nos quedamos con la posición -1
        # target_packet shape: (Batch, hidden_size * 2)
        
        # Classification
        output = self.classifier(target_packet)
        return output.squeeze(-1)

In [ ]:
model_lstm = LoraCollisionLSTM(
    num_features = 8, # 8 reales + 1 Delta_T (este se adiciona en otras funciones)
    hidden_size  = 64, 
    num_layers   = 2, 
    dropout      = 0.1
)

criterion = nn.BCELoss()

optimizer = optim.Adam(model_lstm.parameters(), lr=0.001)


X_train, y_train, X_test, y_test, X_val  , y_val = load_and_prepare_data(seq_length = 16)

train_dataset = TensorDataset(X_train, y_train)
test_dataset  = TensorDataset(X_test, y_test)
val_dataset   = TensorDataset(X_val, y_val)

device = "cpu"
print(f"Using : {device}")
    
train_lstm_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_lstm_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)
val_lstm_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)


EPOCHS = 20
print("Bi-LSTM training...")

for epoch in range(EPOCHS):
    model_lstm.train() 
    train_loss = 0.0
    correct_train = 0
    total_train = 0
    
    # El DataLoader nos da los batches listos mágicamente
    for batch_X, batch_y in train_lstm_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                
        optimizer.zero_grad()                
        predictions = model_lstm(batch_X)                
        loss        = criterion(predictions, batch_y)
                
        loss.backward()                
        optimizer.step()
        
        # Statistics
        train_loss += loss.item() * batch_X.size(0)
        preds_clase = (predictions >= 0.5).float() # Convertir prob a 0 o 1
        correct_train += (preds_clase == batch_y).sum().item()
        total_train += batch_y.size(0)
        
    # Calcular promedio de la fase de entrenamiento
    epoch_train_loss = train_loss / total_train
    epoch_train_acc = correct_train / total_train
    
    # --- FASE DE PRUEBA (EVALUACIÓN) ---
    model_lstm.eval() # Pone al modelo en modo "examen" (desactiva Dropout)
    val_loss = 0.0
    correct_val = 0
    total_val = 0
    
    y_true = []
    y_pred = []
    # torch.no_grad() evita calcular gradientes, haciendo esto muy rápido
    with torch.no_grad():
        for batch_X, batch_y in val_lstm_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            
            # Solo predecimos y calculamos error, NO hacemos .backward() ni .step()
            predictions = model_lstm(batch_X)
            loss = criterion(predictions, batch_y)
            
            # Estadísticas
            val_loss += loss.item() * batch_X.size(0)
            preds_clase = (predictions >= 0.5).float()
            correct_val += (preds_clase == batch_y).sum().item()
            total_val += batch_y.size(0)

            # Other method to calculate metrics
            y_pred.extend(preds_clase.cpu().numpy())  
            y_true.extend(batch_y.cpu().numpy())
            
    # Calcular promedio de la fase de prueba
    epoch_val_loss = val_loss / total_val
    epoch_val_acc = correct_val / total_val
    
    # --- REPORTE DEL EPOCH ---
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    auc = roc_auc_score(y_val, y_pred)
    print(f"Epoch [{epoch+1:02d}/{EPOCHS}] "
          f"Train Loss: {epoch_train_loss:.4f} - Acc: {epoch_train_acc*100:.1f}% || "
          f"Val Loss: {epoch_val_loss:.4f} - Acc: {epoch_val_acc*100:.1f}% ||"
          f"Val Acc: {acc:.4f} - Val Prec: {prec*100:.1f}% | "
          f"Val Rec: {rec:.4f} - Val F1: {f1*100:.1f}% | "
          f"Val AUC-ROC: {auc*100:.4f}"
          )

print("¡Entrenamiento completado!")

In [ ]:
print("\n--- EVALUACIÓN FINAL EN TEST SET ---")
model_lstm.eval() # Pone al modelo en modo "examen" (desactiva Dropout)
test_loss = 0.0
correct_test = 0
total_test = 0

y_true = []
y_pred = []


# torch.no_grad() evita calcular gradientes, haciendo esto muy rápido
with torch.no_grad():
    for batch_X, batch_y in test_lstm_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        # Solo predecimos y calculamos error, NO hacemos .backward() ni .step()
        predictions = model_lstm(batch_X)
        loss = criterion(predictions, batch_y)
        
        # Estadísticas
        test_loss += loss.item() * batch_X.size(0)
        preds_clase = (predictions >= 0.5).float()
        correct_test += (preds_clase == batch_y).sum().item()
        total_test += batch_y.size(0)

        # Other method to calculate metrics
        y_pred.extend(preds_clase.cpu().numpy())  
        y_true.extend(batch_y.cpu().numpy())

        #
        # Guardar para métricas de Scikit-Learn
        ###y_pred_final.extend(predictions.cpu().numpy())   # Probabilidades reales
        ###y_true_final.extend(batch_y.cpu().numpy())
        ###y_pred_clase.extend(preds_clase.cpu().numpy()) # Etiquetas 0 o 1



 # --- REPORTE DEL EPOCH ---
acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, zero_division=0)
rec = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)
auc = roc_auc_score(y_test, y_pred)
print(f"FINAL REPORT"        
        f"Test Acc: {acc:.4f} - Val Prec: {prec*100:.1f}% | "
        f"Test Rec: {rec:.4f} - Val F1: {f1*100:.1f}% | "
        f"Test AUC-ROC: {auc:.4f}"
        )

# GridSearch LSTM

In [9]:
#def train_test(config, train_loader, val_loader, device, epochs=20, save_logs = False):    
def train_test(config, train_loader, val_loader, device, epochs=20):
    logs = {}
    logs['train_loss_log']  = []    
    logs['train_acc_log']   = []
    logs['val_loss_log']    = []
    logs['val_acc_log']     = []

    # Create model with configuration
    model_lstm = LoraCollisionLSTM(        
        num_features = 9, # 8 reales + 1 Delta_T (este se adiciona en otras funciones)
        hidden_size  = config['hidden_size'], #64, 
        num_layers   = config['num_layers'], #2, 
        dropout      = config['dropout'] #0.1
    ).to(device)

        
    criterion = nn.BCELoss()
    #optimizer = optim.Adam(model.parameters(), lr=config['lr'])
    optimizer = optim.Adam(model_lstm.parameters(), lr=0.001)
    
    #best_test_acc = 0.0   
    best_val_acc = 0.0 
    #best_val_loss = float('inf')
    avg_val_loss = 0.0
    for epoch in range(epochs):
        # Train
        model_lstm.train()
        train_loss      = 0.0
        correct_train   = 0
        total_train     = 0

        #for X_b, y_b in train_loader:
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            
            # Forward pass
            predictions = model_lstm(batch_X) # Usa internamente el squeeze(-1)

            # Clean gradient 
            optimizer.zero_grad()

            # Calculate error loss
            loss = criterion(predictions, batch_y)

            # Backpropagation
            loss.backward()

            # Update weights
            optimizer.step()

            # loss.item - The average of the batch losses will give you an estimate of the “epoch loss” during training. Returns the value of this tensor as a standard Python number 
            train_loss += loss.item() * batch_X.size(0)
            
            # Convert to 0 - 1 the vector batch
            predictions_class = (predictions >= 0.5).float()
            
            correct_train += (predictions_class == batch_y).sum().item()
            total_train   += batch_y.size(0)

            
        
        avg_train_loss  = train_loss / total_train
        train_acc       = correct_train / total_train
        
        logs['train_loss_log'].append(avg_train_loss)
        logs['train_acc_log'].append(train_acc)

        # Validation
        model_lstm.eval()        
        val_loss     = 0.0  
        correct_val = 0
        total_val   = 0
        
        y_true = []
        y_pred = []
        with torch.no_grad():
            #for X_b, y_b in val_loader:
            for batch_X, batch_y in val_loader:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                
                # Predict
                predictions = model_lstm(batch_X)
                loss        = criterion(predictions, batch_y)
                val_loss   += loss.item() * batch_X.size(0)

                # Calculate correct or not 
                predictions_class = (predictions >= 0.5).float()
                correct_val      += (predictions_class == batch_y).sum().item()
                total_val        += batch_y.size(0)

                # Other method to calculate metrics
                y_pred.extend(predictions_class.cpu().numpy())  
                y_true.extend(batch_y.cpu().numpy())
                
        
        avg_val_loss = val_loss / total_val
        val_acc      = correct_val / total_val

        
        logs['val_loss_log'].append(avg_val_loss)
        logs['val_acc_log'].append(val_acc)
        
        # Print metrics
        acc = accuracy_score(y_true, y_pred)
        prec = precision_score(y_true, y_pred, zero_division=0)
        rec = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        auc = roc_auc_score(y_true, y_pred)
        

        print(f"Epoch {epoch+1:02d}/{epochs} | "
        f"Train Loss: {avg_train_loss:.4f} - Acc: {train_acc*100:.1f}% | "
        f"Val Loss: {avg_val_loss:.4f} - Acc: {val_acc*100:.1f}% | " 
        f"Val Acc: {acc:.4f} - Val Prec: {prec*100:.1f}% | "
        f"Val Rec: {rec:.4f} - Val F1: {f1*100:.1f}% | "
        f"Val AUC-ROC: {auc:.4f} "
        )

    
    return avg_val_loss, model_lstm, logs

In [10]:
def fine_tuning(train_dataset, test_dataset, val_dataset):
    device = "cpu"
    print(f"Using : {device}")    
    #device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # Setting params    
    param_grid = {
        'hidden_size'   : [32, 64], #128                # Dimension of the mode
        'num_layers'  : [1, 2, 3],                    # Number of encoder layers
        'dropout'   : [0.1, 0.2], # , 0.2                # doprout
        'batch_size': [32]                          # Batch Size
    }


    # Generate the combination of parameters
    keys, values = zip(*param_grid.items())
    combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]
    
    print(f"Starting  {len(combinations)} combinations ...")
    print("="*60)
    
    #print(combinations)

    results     = []    
    start_total = time.time()  
    best_val_loss = float('inf')  
    best_logs = None
    best_config = None
    name_model = ''
    for i, config in enumerate(combinations):
        # Create DataLoaders
        train_loader = DataLoader(train_dataset, batch_size=config['batch_size'], shuffle=True)
        test_loader  = DataLoader(test_dataset, batch_size=config['batch_size'], shuffle=False)
        val_loader   = DataLoader(val_dataset, batch_size=config['batch_size'], shuffle=False)
        
        print(f"[{i+1}/{len(combinations)}] Testing: {config} ...", end=" ")
                
        # Training
        final_loss, model_lstm, logs = train_test(config, train_loader, val_loader, device, epochs=20)
        
        print(f"-> Best Loss: {final_loss:.4f}")
            
        if final_loss < best_val_loss:
            print(f"New best model found with val_loss: {final_loss:.4f} (previous best: {best_val_loss:.4f}). Saving model...")
            name_model = 'LSTM_' + str(config['hidden_size']) + '_' + str(config['num_layers']) + '_loss_' + str(round(final_loss, 4)) + '.pth'
            best_val_loss = final_loss
            best_logs = logs
            best_config = config
            torch.save(model_lstm.state_dict(), name_model)
            print("Model saved !!!")


        # Guardar resultado
        res             = config.copy()
        res['loss'] = final_loss
        results.append(res)
        #'''
    
    print("="*60)
    print(f"Finishing in {(time.time() - start_total)/60:.1f} minutes.")
        
    #results_df = pd.DataFrame(results)
    #results_df = results_df.sort_values(by='accuracy', ascending=False)
    
    #print("Best results")
    #print(results_df.head(5))
    
    # Save to CSV
    #results_df.to_csv('hyperparameter_results.csv', index=False)
    
    
    # Return best configuration
    #best_config = config#.to_dict() #results_df.iloc[0].to_dict()
    return best_config, name_model, best_logs

In [11]:
X_train, y_train, X_test, y_test, X_val  , y_val = load_and_prepare_data(seq_length = 16)

train_dataset = TensorDataset(X_train, y_train)
test_dataset  = TensorDataset(X_test, y_test)
val_dataset   = TensorDataset(X_val, y_val)

''' 
device = "cpu"
print(f"Using : {device}")
    
train_lstm_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_lstm_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)
val_lstm_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)
'''

Loading dataset...


/home/ylnner/miniconda3/envs/p_new/lib/python3.14/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/home/ylnner/miniconda3/envs/p_new/lib/python3.14/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/home/ylnner/miniconda3/envs/p_new/lib/python3.14/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


' \ndevice = "cpu"\nprint(f"Using : {device}")\n\ntrain_lstm_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)\ntest_lstm_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)\nval_lstm_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)\n'

In [ ]:
X_train.shape

In [12]:
best_params, best_name_model, best_logs = fine_tuning(train_dataset, test_dataset, val_dataset)
print("Best configuration : ")
print(best_params)
print(f"Best model name: {best_name_model}")

Using : cpu
Starting  12 combinations ...
[1/12] Testing: {'hidden_size': 32, 'num_layers': 1, 'dropout': 0.1, 'batch_size': 32} ... 

/home/ylnner/miniconda3/envs/p_new/lib/python3.14/site-packages/torch/nn/modules/rnn.py:990: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.1 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)


Epoch 01/20 | Train Loss: 0.4956 - Acc: 77.1% | Val Loss: 0.2986 - Acc: 88.2% | Val Acc: 0.8822 - Val Prec: 81.0% | Val Rec: 0.8825 - Val F1: 84.4% | Val AUC-ROC: 0.8823 
Epoch 02/20 | Train Loss: 0.3470 - Acc: 83.8% | Val Loss: 0.2287 - Acc: 89.5% | Val Acc: 0.8949 - Val Prec: 84.5% | Val Rec: 0.8700 - Val F1: 85.7% | Val AUC-ROC: 0.8895 
Epoch 03/20 | Train Loss: 0.3105 - Acc: 85.4% | Val Loss: 0.2060 - Acc: 91.7% | Val Acc: 0.9167 - Val Prec: 83.8% | Val Rec: 0.9550 - Val F1: 89.3% | Val AUC-ROC: 0.9249 
Epoch 04/20 | Train Loss: 0.2946 - Acc: 87.1% | Val Loss: 0.1899 - Acc: 93.1% | Val Acc: 0.9312 - Val Prec: 88.9% | Val Rec: 0.9250 - Val F1: 90.7% | Val AUC-ROC: 0.9298 
Epoch 05/20 | Train Loss: 0.2783 - Acc: 87.6% | Val Loss: 0.1808 - Acc: 93.2% | Val Acc: 0.9321 - Val Prec: 87.9% | Val Rec: 0.9425 - Val F1: 91.0% | Val AUC-ROC: 0.9343 
Epoch 06/20 | Train Loss: 0.2676 - Acc: 88.4% | Val Loss: 0.1784 - Acc: 92.6% | Val Acc: 0.9257 - Val Prec: 88.8% | Val Rec: 0.9100 - Val F1: 89.

/home/ylnner/miniconda3/envs/p_new/lib/python3.14/site-packages/torch/nn/modules/rnn.py:990: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)


Epoch 01/20 | Train Loss: 0.5069 - Acc: 74.6% | Val Loss: 0.2987 - Acc: 87.4% | Val Acc: 0.8741 - Val Prec: 83.5% | Val Rec: 0.8125 - Val F1: 82.4% | Val AUC-ROC: 0.8608 
Epoch 02/20 | Train Loss: 0.3501 - Acc: 84.0% | Val Loss: 0.2239 - Acc: 91.3% | Val Acc: 0.9130 - Val Prec: 89.4% | Val Rec: 0.8625 - Val F1: 87.8% | Val AUC-ROC: 0.9021 
Epoch 03/20 | Train Loss: 0.3067 - Acc: 86.4% | Val Loss: 0.1990 - Acc: 92.3% | Val Acc: 0.9230 - Val Prec: 88.9% | Val Rec: 0.9000 - Val F1: 89.4% | Val AUC-ROC: 0.9180 
Epoch 04/20 | Train Loss: 0.2869 - Acc: 87.6% | Val Loss: 0.1812 - Acc: 93.3% | Val Acc: 0.9330 - Val Prec: 87.0% | Val Rec: 0.9575 - Val F1: 91.2% | Val AUC-ROC: 0.9383 
Epoch 05/20 | Train Loss: 0.2787 - Acc: 87.8% | Val Loss: 0.1729 - Acc: 93.7% | Val Acc: 0.9366 - Val Prec: 88.9% | Val Rec: 0.9425 - Val F1: 91.5% | Val AUC-ROC: 0.9379 
Epoch 06/20 | Train Loss: 0.2655 - Acc: 88.3% | Val Loss: 0.1711 - Acc: 93.2% | Val Acc: 0.9321 - Val Prec: 89.7% | Val Rec: 0.9175 - Val F1: 90.

/home/ylnner/miniconda3/envs/p_new/lib/python3.14/site-packages/torch/nn/modules/rnn.py:990: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.1 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)


Epoch 01/20 | Train Loss: 0.4751 - Acc: 77.0% | Val Loss: 0.2769 - Acc: 88.1% | Val Acc: 0.8813 - Val Prec: 83.9% | Val Rec: 0.8325 - Val F1: 83.6% | Val AUC-ROC: 0.8708 
Epoch 02/20 | Train Loss: 0.3341 - Acc: 84.7% | Val Loss: 0.2070 - Acc: 92.1% | Val Acc: 0.9212 - Val Prec: 87.9% | Val Rec: 0.9075 - Val F1: 89.3% | Val AUC-ROC: 0.9182 
Epoch 03/20 | Train Loss: 0.2976 - Acc: 86.8% | Val Loss: 0.1849 - Acc: 93.7% | Val Acc: 0.9366 - Val Prec: 88.2% | Val Rec: 0.9525 - Val F1: 91.6% | Val AUC-ROC: 0.9400 
Epoch 04/20 | Train Loss: 0.2789 - Acc: 87.7% | Val Loss: 0.1753 - Acc: 93.4% | Val Acc: 0.9339 - Val Prec: 90.4% | Val Rec: 0.9150 - Val F1: 90.9% | Val AUC-ROC: 0.9298 
Epoch 05/20 | Train Loss: 0.2632 - Acc: 88.2% | Val Loss: 0.1676 - Acc: 93.9% | Val Acc: 0.9393 - Val Prec: 89.7% | Val Rec: 0.9400 - Val F1: 91.8% | Val AUC-ROC: 0.9395 
Epoch 06/20 | Train Loss: 0.2474 - Acc: 89.2% | Val Loss: 0.1553 - Acc: 93.9% | Val Acc: 0.9393 - Val Prec: 89.2% | Val Rec: 0.9475 - Val F1: 91.

/home/ylnner/miniconda3/envs/p_new/lib/python3.14/site-packages/torch/nn/modules/rnn.py:990: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)


Epoch 01/20 | Train Loss: 0.4855 - Acc: 76.3% | Val Loss: 0.2852 - Acc: 86.4% | Val Acc: 0.8641 - Val Prec: 84.7% | Val Rec: 0.7625 - Val F1: 80.3% | Val AUC-ROC: 0.8422 
Epoch 02/20 | Train Loss: 0.3430 - Acc: 84.4% | Val Loss: 0.2237 - Acc: 92.3% | Val Acc: 0.9230 - Val Prec: 88.3% | Val Rec: 0.9075 - Val F1: 89.5% | Val AUC-ROC: 0.9197 
Epoch 03/20 | Train Loss: 0.3026 - Acc: 86.4% | Val Loss: 0.1936 - Acc: 93.1% | Val Acc: 0.9312 - Val Prec: 88.6% | Val Rec: 0.9300 - Val F1: 90.7% | Val AUC-ROC: 0.9309 
Epoch 04/20 | Train Loss: 0.2843 - Acc: 87.5% | Val Loss: 0.1814 - Acc: 93.4% | Val Acc: 0.9339 - Val Prec: 91.0% | Val Rec: 0.9075 - Val F1: 90.9% | Val AUC-ROC: 0.9282 
Epoch 05/20 | Train Loss: 0.2689 - Acc: 88.3% | Val Loss: 0.1760 - Acc: 93.0% | Val Acc: 0.9303 - Val Prec: 90.7% | Val Rec: 0.9000 - Val F1: 90.3% | Val AUC-ROC: 0.9237 
Epoch 06/20 | Train Loss: 0.2507 - Acc: 89.3% | Val Loss: 0.1617 - Acc: 93.5% | Val Acc: 0.9348 - Val Prec: 89.8% | Val Rec: 0.9250 - Val F1: 91.

In [14]:
best_name_model

'LSTM_64_3_loss_0.0546.pth'

In [15]:
best_params

{'hidden_size': 64, 'num_layers': 3, 'dropout': 0.1, 'batch_size': 32}

In [16]:
best_name_model = best_name_model


print("\n--- EVALUACIÓN FINAL EN TEST SET ---")
device = "cpu"
# Instanciar modelo con la configuración actual
''' 
final_model = LoraCollisionLSTM(
    num_numerical_features  = 8, # 8 reales + 1 t_delta (el delta se adiciona despues)
    d_model                 = best_params['d_model'],
    n_heads                 = best_params['n_heads'],
    n_layers                = best_params['n_layers'],
    dropout                 = best_params['dropout']
).to(device)
'''

final_model = LoraCollisionLSTM(        
    num_features = 9, # 8 reales + 1 Delta_T (este se adiciona en otras funciones)
    hidden_size  = best_params['hidden_size'], #64, 
    num_layers   = best_params['num_layers'], #2, 
    dropout      = best_params['dropout'] #0.1
).to(device)

#name_to_load = 'Transformer_32_4_loss_0.0882.pth'


test_loader  = DataLoader(test_dataset, batch_size=best_params['batch_size'], shuffle=False)

# 2. Cargas los pesos guardados (.pth)
final_model.load_state_dict(torch.load(best_name_model))
final_model.eval() # Modo evaluación 

criterion = nn.BCELoss()

#optimizer = optim.Adam(model_lstm.parameters(), lr=0.001)

# 3. Evaluación final (Una sola pasada)
test_loss = 0.0
correct_test = 0
total_test = 0
y_true, y_pred = [], []
with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)

        # Solo predecimos y calculamos error, NO hacemos .backward() ni .step()
        predictions = final_model(batch_X)
        loss = criterion(predictions, batch_y)
        
        # Estadísticas
        test_loss += loss.item() * batch_X.size(0)
        preds_clase = (predictions >= 0.5).float()
        correct_test += (preds_clase == batch_y).sum().item()
        total_test += batch_y.size(0)

        # Other method to calculate metrics
        y_pred.extend(preds_clase.cpu().numpy())  
        y_true.extend(batch_y.cpu().numpy())


 # --- REPORTE DEL EPOCH ---
acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, zero_division=0)
rec = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)
auc = roc_auc_score(y_test, y_pred)
print(f"FINAL REPORT"        
        f"Test Acc: {acc:.4f} - Val Prec: {prec*100:.1f}% | "
        f"Test Rec: {rec:.4f} - Val F1: {f1*100:.1f}% | "
        f"Test AUC-ROC: {auc:.4f}"
        )


print(f"F1 Final en Test: {f1_score(y_true, y_pred):.4f}")


--- EVALUACIÓN FINAL EN TEST SET ---
FINAL REPORTTest Acc: 0.9547 - Val Prec: 93.9% | Test Rec: 0.9649 - Val F1: 95.2% | Test AUC-ROC: 0.9554
F1 Final en Test: 0.9519


In [ ]:

Transformer
Test Loss: 0.1517 - Acc: 93.9% | Test Acc: 0.9393 -
Test Prec: 92.7% | 
Test Rec: 0.9241 - 
Test F1: 92.5% | 
Test AUC-ROC: 93.6907 

Bi-LSTM
Test Acc: 0.9377 - 
Test Prec: 89.0% | 
Test Rec: 0.9660 - 
Test F1: 92.7% | 
Test AUC-ROC: 94.2140


Model: Logistic Regression
Accuracy:  81.94%
Precision: 79.56%
Recall:    74.04%
F1-Score:  76.70%
AUC-ROC:  80.64%

Model: Random Forest
Accuracy:  91.59%
Precision: 87.73%
Recall:    91.90%
F1-Score:  89.77%
AUC-ROC:  91.64%

Model: MLP
Accuracy:  91.18%
Precision: 88.66%
Recall:    89.46%
F1-Score:  89.06%
AUC-ROC:  90.89%

Mathematic
Testing...
-> Accuracy:  0.8736
-> Precision: 0.7620
-> Recall:    0.9961
-> F1-Score:  0.8635
-> AUC-ROC:  0.9206

In [17]:
total_params = sum(p.numel() for p in final_model.parameters())
print(f"Total parameters: {total_params}")

Total parameters: 241217


# LSTM

In [ ]:
model_no_bi_lstm = LoraCollisionLSTM(
    num_features = 9, # 8 reales + 1 Delta_T
    hidden_size  = 64, 
    num_layers   = 2, 
    dropout      = 0.1, 
    is_bidirectional=False
)

criterion = nn.BCELoss()

optimizer = optim.Adam(model_no_bi_lstm.parameters(), lr=0.001)

In [ ]:
X_train, y_train, X_test, y_test, X_val  , y_val = load_and_prepare_data(seq_length = 16)

train_dataset = TensorDataset(X_train, y_train)
test_dataset  = TensorDataset(X_test, y_test)
val_dataset   = TensorDataset(X_val, y_val)

# Dataset Split
#dataset = TensorDataset(X, y)

#train_size  = int(0.8 * len(dataset))
#test_size   = len(dataset) - train_size
#generator1  = torch.Generator().manual_seed(42)

#train_data, test_data = random_split(dataset, [train_size, test_size], generator=generator1)

#device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = "cpu"
print(f"Using : {device}")
    
# Train and test with best params
#train_lstm_loader = DataLoader(train_data, batch_size=32, shuffle=True)
#test_lstm_loader  = DataLoader(test_data, batch_size=32, shuffle=False)


train_lstm_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_lstm_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)
val_lstm_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)



In [ ]:
EPOCHS = 20
print("LSTM training...")

for epoch in range(EPOCHS):
    model_no_bi_lstm.train() 
    train_loss = 0.0
    correct_train = 0
    total_train = 0
    
    # El DataLoader nos da los batches listos mágicamente
    for batch_X, batch_y in train_lstm_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                
        optimizer.zero_grad()                
        predictions = model_no_bi_lstm(batch_X)                
        loss        = criterion(predictions, batch_y)
                
        loss.backward()                
        optimizer.step()
        
        # Statistics
        train_loss += loss.item() * batch_X.size(0)
        preds_clase = (predictions >= 0.5).float() # Convertir prob a 0 o 1
        correct_train += (preds_clase == batch_y).sum().item()
        total_train += batch_y.size(0)
        
    # Calcular promedio de la fase de entrenamiento
    epoch_train_loss = train_loss / total_train
    epoch_train_acc = correct_train / total_train
    
    # --- FASE DE PRUEBA (EVALUACIÓN) ---
    model_no_bi_lstm.eval() # Pone al modelo en modo "examen" (desactiva Dropout)
    val_loss = 0.0
    correct_val = 0
    total_val = 0
    
    y_true = []
    y_pred = []
    # torch.no_grad() evita calcular gradientes, haciendo esto muy rápido
    with torch.no_grad():
        for batch_X, batch_y in val_lstm_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            
            # Solo predecimos y calculamos error, NO hacemos .backward() ni .step()
            predictions = model_no_bi_lstm(batch_X)
            loss = criterion(predictions, batch_y)
            
            # Estadísticas
            val_loss += loss.item() * batch_X.size(0)
            preds_clase = (predictions >= 0.5).float()
            correct_val += (preds_clase == batch_y).sum().item()
            total_val += batch_y.size(0)

            # Other method to calculate metrics
            y_pred.extend(preds_clase.cpu().numpy())  
            y_true.extend(batch_y.cpu().numpy())
            
    # Calcular promedio de la fase de prueba
    epoch_test_loss = val_loss / total_val
    epoch_test_acc = correct_val / total_val
    
    # --- REPORTE DEL EPOCH ---
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    auc = roc_auc_score(y_val, y_pred)
    print(f"Epoch [{epoch+1:02d}/{EPOCHS}] "
          f"Train Loss: {epoch_train_loss:.4f} - Acc: {epoch_train_acc*100:.1f}% || "
          f"Val Loss: {epoch_test_loss:.4f} - Acc: {epoch_test_acc*100:.1f}% ||"
          f"Val Acc: {acc:.4f} - Val Prec: {prec*100:.1f}% | "
          f"Val Rec: {rec:.4f} - Val F1: {f1*100:.1f}% | "
          f"Val AUC-ROC: {auc:.4f}"
          )

print("¡Entrenamiento completado!")

In [ ]:
print("\n--- EVALUACIÓN FINAL EN TEST SET ---")
model_no_bi_lstm.eval() # Pone al modelo en modo "examen" (desactiva Dropout)
test_loss = 0.0
correct_test = 0
total_test = 0

y_true = []
y_pred = []


# torch.no_grad() evita calcular gradientes, haciendo esto muy rápido
with torch.no_grad():
    for batch_X, batch_y in test_lstm_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        # Solo predecimos y calculamos error, NO hacemos .backward() ni .step()
        predictions = model_no_bi_lstm(batch_X)
        loss = criterion(predictions, batch_y)
        
        # Estadísticas
        test_loss += loss.item() * batch_X.size(0)
        preds_clase = (predictions >= 0.5).float()
        correct_test += (preds_clase == batch_y).sum().item()
        total_test += batch_y.size(0)

        # Other method to calculate metrics
        y_pred.extend(preds_clase.cpu().numpy())  
        y_true.extend(batch_y.cpu().numpy())

        #
        # Guardar para métricas de Scikit-Learn
        ###y_pred_final.extend(predictions.cpu().numpy())   # Probabilidades reales
        ###y_true_final.extend(batch_y.cpu().numpy())
        ###y_pred_clase.extend(preds_clase.cpu().numpy()) # Etiquetas 0 o 1



 # --- REPORTE DEL EPOCH ---
acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, zero_division=0)
rec = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)
auc = roc_auc_score(y_test, y_pred)
print(f"FINAL REPORT"        
        f"Test Acc: {acc:.4f} - Val Prec: {prec*100:.1f}% | "
        f"Test Rec: {rec:.4f} - Val F1: {f1*100:.1f}% | "
        f"Test AUC-ROC: {auc:.4f}"
        )